# Constellation v2 — Detector training on Colab

From-scratch hand+head detector (Stage 1). Runs the SAME `aslv2.detect.train` code as local; only the device (CUDA) and data root differ.

## Before running — upload these to one Drive folder (default `MyDrive/asl-detector/`):
1. `colab_detector_code.zip`  (from `model-v2/artifacts/`, run `python scripts/package_for_colab.py` to make it) — code + manifests
2. `raw.zip`            (from `model-v2/data/detect/100doh/`)        — 100DOH frames (~8.8 GB)
3. `WIDER_train.zip`    (from `model-v2/data/detect/widerface/`)     — WIDER train images (~1.4 GB)
4. `WIDER_val.zip`      (from `model-v2/data/detect/widerface/`)     — WIDER val images (~0.35 GB)

You do **not** need the WIDER annotations or 100DOH `file.json` — the manifests already contain the boxes.

**Runtime → Change runtime type → GPU (A100/L4/T4).**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR  = '/content/drive/MyDrive/asl-detector'   # <-- the folder you uploaded the 4 files to
DATA_ROOT  = '/content/data/detect'
CODE_DIR   = '/content/model-v2'
assert os.path.isdir(DRIVE_DIR), f'Upload the 4 files to {DRIVE_DIR} first'
print('Drive folder contents:', os.listdir(DRIVE_DIR))

In [ ]:
# Copy zips from Drive to fast local Colab disk, then extract into the layout
# that matches the manifests' relative paths (100doh/raw/..., widerface/WIDER_*/...).
import shutil
os.makedirs(f'{DATA_ROOT}/100doh', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/widerface', exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

for z in ['raw.zip', 'WIDER_train.zip', 'WIDER_val.zip', 'colab_detector_code.zip']:
    src = f'{DRIVE_DIR}/{z}'
    print('copying', z, '...')
    shutil.copy(src, f'/content/{z}')
print('copied all zips to /content')

In [ ]:
# Extract (raw.zip -> 100doh/raw/...,  WIDER_*.zip -> widerface/WIDER_*/...)
!unzip -q -o /content/raw.zip          -d $DATA_ROOT/100doh/
!unzip -q -o /content/WIDER_train.zip  -d $DATA_ROOT/widerface/
!unzip -q -o /content/WIDER_val.zip    -d $DATA_ROOT/widerface/
!unzip -q -o /content/colab_detector_code.zip -d $CODE_DIR

# sanity: a manifest-relative path must resolve under DATA_ROOT
import json
m = json.load(open(f'{CODE_DIR}/artifacts/detect/val.json'))
p = os.path.join(DATA_ROOT, m[0]['image'])
print('sample image resolves:', os.path.exists(p), '->', p)

In [ ]:
!pip install -q -e $CODE_DIR
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# Train. device() auto-selects CUDA on Colab. Checkpoints land in artifacts/checkpoints/detector/.
!cd $CODE_DIR && python -m aslv2.detect.train --config configs/detector.yaml --data-root $DATA_ROOT

In [ ]:
# Persist the trained detector back to Drive (so it survives the session).
ckpt = f'{CODE_DIR}/artifacts/checkpoints/detector'
for f in ['best.pt', 'history.json']:
    src = f'{ckpt}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_DIR}/{f}')
        print('saved to Drive:', f)

import json
h = json.load(open(f'{ckpt}/history.json'))
print('best val:', h.get('best_score'), '| final epoch metrics:', h['history'][-1] if h.get('history') else None)
print('\nDownload best.pt from Drive into model-v2/artifacts/checkpoints/detector/ on your machine.')